# Part 4 — Surviving a Broken Plan: the prospective critic and error-triggered replanning

*Agents from First Principles, Part 4.*

Part 3 made the plan a **first-class artifact**: write the whole DAG down once, then execute it cheaply. That is a real win, and it buys a new failure. A plan written up front is a bet that the world will still look the way it did when you planned. The instant the world disagrees, a committed plan is a liability, and the DAG executor from Part 3 charges ahead and executes it anyway.

The break we inject is the realistic one: a SKU is **DISCONTINUED**. The plan was built to quote a price and warranty for `SKU-ACME-EB`, but mid-run that product turns out to be retired and replaced by `SKU-GLX-EB`. Part 2 already taught us to classify the failure: a discontinued SKU is a **PERMANENT** error (retrying the same call cannot help). Part 2 fed that error back as an observation. That is necessary and NOT sufficient here, because the agent is now committed to a plan that names a dead SKU in three places. Knowing the step failed is not the same as being able to do something about a plan you already wrote.

Two mechanisms fix it, one before execution and one during:

1. **A PROSPECTIVE CRITIC.** Before a single tool fires, check the plan for the structural mistakes a planner makes: an **UNKNOWN TOOL** (not in the registry), an **UNSATISFIABLE DEP** (a step that needs a node that does not exist), a **DEPENDENCY CYCLE** (a step that transitively needs itself), and a **REDUNDANT** step (two nodes with the identical tool and argument). A bad plan is rejected before it wastes a single call. This is the plan-time analog of Part 1's pre-flight argument validator.
2. **An ERROR-TRIGGERED REPLANNER.** When a step fails in a way that invalidates the rest of the plan, do not abandon the run and do not start over. Revise ONLY THE REMAINING SUBGRAPH: rewrite the not-yet-run steps to target the replacement, keep every completed step **MEMOIZED** (so it is never re-run), and continue. An honest **REPLAN BUDGET** caps how many times this can happen, so a plan that keeps breaking trips out instead of looping forever (the full circuit breaker is Part 8).

The contrast we show: a **BLIND** executor (Part 3, no critic, no replanner) hits the discontinued SKU, dutifully feeds back the permanent error, and still cannot produce a valid quote, because it has no way to revise the committed plan. The **CRITIC + REPLANNER** executor clears the plan up front, hits the same failure, rewrites only the dead tail to the replacement SKU, reuses the memoized lookup, and finishes with a correct quote.

**REUSE, not rebuild**: the `TaskNode`, the dependency DAG, and topological execution come straight from Part 3. This part adds the critic and the replanner around them.

**Continuity**: same Acme to Globex world (the acquisition retires the Acme earbuds in favor of the Globex line). Deterministic rule critic/planner offline; `generate()` is the real-LLM path one env flag away.

> **Runs with no network, no API key, and no third-party dependency.** Set `OPENAI_API_KEY` to see the real-planner banner; the critic and replanner always fall through to the deterministic rules so output stays reproducible.

Companion script: `replanning_critic.py`

## Setup

Two standard-library imports do the work: `os` lets us check for an API key without ever requiring one, and `re` powers the calculator's input guard. Nothing in the default path is imported from a third-party package, so every cell runs fully offline, exactly as in Parts 1-3.

In [ ]:
import os
import re

print("ready -- no network, no API key, no third-party dependency required")

## Step 0 -- The world: a tiny catalog where one SKU is discontinued

The world is just two products. `SKU-ACME-EB`, the Acme earbuds, is **discontinued** and points at its replacement, `SKU-GLX-EB`, the Globex earbuds (price `$79.00`, a 2-year limited warranty). This is the Acme to Globex acquisition from Parts 1-3 made concrete at the catalog level: the acquisition retires the Acme line in favor of the Globex one. The whole drama of this part lives in that one `discontinued` record, because the plan was written to quote the dead SKU.

In [ ]:
# The Acme -> Globex acquisition made concrete: the Acme earbuds are retired in
# favor of the Globex line, and the dead SKU points at its replacement.
CATALOG = {
    "SKU-ACME-EB": {"status": "discontinued", "replacement": "SKU-GLX-EB"},
    "SKU-GLX-EB": {"status": "active", "price": 79.0, "warranty": "2-year limited warranty"},
}

print(f"Catalog: {len(CATALOG)} SKUs. SKU-ACME-EB is discontinued, replaced by SKU-GLX-EB.")

## Step 0b -- `DiscontinuedError`: a PERMANENT error that carries its replacement

In Part 2's taxonomy a discontinued SKU is a **permanent** failure: retrying the same call cannot help, so there is no point feeding it through a retry loop. Two details make it useful to the replanner. First, it is a distinct exception type, so the executor can catch *this* failure specifically and treat it as plan-invalidating rather than transient. Second, it carries the `replacement` SKU on the exception itself, so when the replanner catches it, it already knows where to point the dead tail. The error is not just a complaint; it is a pointer to the fix.

In [ ]:
class DiscontinuedError(Exception):
    """A PERMANENT error (Part 2's taxonomy): retrying the same SKU cannot help.
    Carries the replacement so the replanner knows where to point the dead tail."""

    def __init__(self, sku, replacement):
        super().__init__(f"{sku} is discontinued (replaced by {replacement})")
        self.sku = sku
        self.replacement = replacement


print("DiscontinuedError ready: a permanent error that names its own replacement.")

## Step 1 -- The tools and the `REGISTRY`

Four tools, and the failure is wired into them. `lookup_status` is informational: it reports whether a SKU is live and, for a dead one, names the replacement, but it never raises. `get_price` and `get_warranty` **REFUSE** a discontinued SKU by raising `DiscontinuedError`: asking the price of a retired product is the permanent failure, not a transient hiccup. `calculator` is the guarded `eval` carried from Parts 1-3, and it returns a readable error string (rather than raising) when its expression still contains an unresolved reference, which is exactly what happens to the blind executor's tax step. `REGISTRY` is the action space the critic checks plans against: a tool not in this dict is an UNKNOWN TOOL.

In [ ]:
def lookup_status(sku):
    rec = CATALOG.get(sku, {"status": "unknown"})
    if rec["status"] == "discontinued":
        return f"{sku}: discontinued, replaced by {rec['replacement']}"
    return f"{sku}: {rec['status']}"


def get_price(sku):
    rec = CATALOG.get(sku, {"status": "unknown"})
    if rec.get("status") == "discontinued":               # REFUSE a dead SKU
        raise DiscontinuedError(sku, rec["replacement"])
    return rec.get("price")


def get_warranty(sku):
    rec = CATALOG.get(sku, {"status": "unknown"})
    if rec.get("status") == "discontinued":               # REFUSE a dead SKU
        raise DiscontinuedError(sku, rec["replacement"])
    return rec.get("warranty")


_CALC_RE = re.compile(r"^[\d\s+\-*/().]+$")


def calculator(expression):
    if not _CALC_RE.match(expression):                    # e.g. an unresolved #E2
        return "calculator error: cannot evaluate (missing input?)"
    try:
        return eval(expression, {"__builtins__": {}}, {})  # guarded: digits/ops only
    except Exception as exc:
        return f"calculator error: {type(exc).__name__}"


# REGISTRY is the action space the critic checks every plan against.
REGISTRY = {
    "lookup_status": lookup_status,
    "get_price": get_price,
    "get_warranty": get_warranty,
    "calculator": calculator,
}

print("Tools defined:", ", ".join(REGISTRY))

## Step 2 -- The plan as data: `TaskNode`, `topo_order`, `dag_levels`, `_bind_arg` (carried from Part 3)

Nothing here is new; it is the DAG machinery from Part 3, restated so the notebook is self-contained. A `TaskNode` is one planned tool call: an id (`E1`, `E2`, ...), the tool, an argument (which may carry an `#En` reference to another node's result), and an explicit list of dependency ids. `topo_order` returns a stable topological order so a node always runs after its dependencies. `dag_levels` lays nodes out by longest-path depth, so independent nodes share a level (a round). `_bind_arg` resolves `#En` references using already-computed results, turning `'#E2 * 1.18'` into `'79.0 * 1.18'` once `E2` has run. We do **not** rebuild this; the critic and replanner are built around it.

In [ ]:
class TaskNode:
    def __init__(self, eid, tool, arg, deps):
        self.eid = eid          # "E1", "E2", ...
        self.tool = tool
        self.arg = arg          # may contain an "#En" reference
        self.deps = deps        # list of eids this node needs first
        self.result = None      # filled at execution


def topo_order(plan):
    """Stable topological order: a node appears after all its dependencies."""
    by_id = {n.eid: n for n in plan}
    done, order = set(), []

    def visit(n):
        if n.eid in done:
            return
        for d in n.deps:
            if d in by_id:
                visit(by_id[d])
        done.add(n.eid)
        order.append(n)

    for n in plan:
        visit(n)
    return order


def dag_levels(plan):
    by_id = {n.eid: n for n in plan}
    level = {}

    def lvl(eid):
        if eid in level:
            return level[eid]
        level[eid] = 1 + max([lvl(d) for d in by_id[eid].deps if d in by_id], default=0)
        return level[eid]

    for n in plan:
        lvl(n.eid)
    return level


def _bind_arg(arg, memo):
    """Resolve #En references using already-computed results (Part 3's binding)."""
    out = arg
    for eid, value in memo.items():
        token = "#" + eid
        if token in out and not isinstance(value, dict):
            out = out.replace(token, f"{value}")
    return out


print("TaskNode, topo_order, dag_levels, _bind_arg ready (carried from Part 3).")

## Step 2b -- The goal and the rule planner

The goal is a quote for the Acme earbuds: price, warranty, and a tax-included total. `rule_planner` is the deterministic offline planner; a real system swaps its body for one `generate()` call returning the same `TaskNode` shape. For a given SKU it emits four nodes: `E1` looks up status, `E2` gets the price, `E3` gets the warranty, and `E4` computes the tax-included total. The only dependency is `E4` needing `E2` (the total is `#E2 * 1.18`), so `E1`, `E2`, `E3` sit at level 1 and `E4` at level 2. Note that `E4`'s argument names a result variable, not a SKU, which is why the replanner will later leave it out of the rewritten tail.

In [ ]:
GOAL = "Quote the price, warranty, and tax-included total for the Acme earbuds (SKU-ACME-EB)."


def rule_planner(sku):
    """Plan a quote for `sku`. E4 (the total) depends on E2 (the price)."""
    return [
        TaskNode("E1", "lookup_status", sku, deps=[]),
        TaskNode("E2", "get_price", sku, deps=[]),
        TaskNode("E3", "get_warranty", sku, deps=[]),
        TaskNode("E4", "calculator", "#E2 * 1.18", deps=["E2"]),
    ]


print("GOAL and rule_planner ready.")
print("Plan:", [(n.eid, n.tool, n.deps) for n in rule_planner("SKU-ACME-EB")])

## Step 3 -- The PROSPECTIVE CRITIC (with `_has_cycle`)

This is the first net-new mechanism. Before any tool fires, the critic inspects the plan for the four structural mistakes a planner makes and returns a list of problems; an empty list means the plan is **cleared for execution**. This is the plan-time analog of Part 1's pre-flight argument validator: catch the broken plan before it wastes a single call.

The four checks: an **UNKNOWN TOOL** (a node whose tool is not in `REGISTRY`), an **UNSATISFIABLE DEP** (a node that depends on an id not present in the plan), a **REDUNDANT** step (two nodes with the identical `(tool, arg)` pair, the second flagged against the first), and a **DEPENDENCY CYCLE** (a step that transitively depends on itself). The cycle check is its own helper, `_has_cycle`, a three-color depth-first search: a back-edge to a node still on the recursion stack (color 1) is a cycle.

In [ ]:
def _has_cycle(plan):
    by_id = {n.eid: n for n in plan}
    color = {}                                  # 0 unseen, 1 on-stack, 2 done

    def dfs(eid):
        if eid not in by_id:
            return False
        color[eid] = 1
        for d in by_id[eid].deps:
            if color.get(d, 0) == 1:            # back-edge to a node on the stack
                return True
            if color.get(d, 0) == 0 and dfs(d):
                return True
        color[eid] = 2
        return False

    return any(color.get(n.eid, 0) == 0 and dfs(n.eid) for n in plan)


def critic(plan, registry):
    problems = []
    ids = {n.eid for n in plan}
    seen = {}
    for n in plan:
        if n.tool not in registry:                                  # UNKNOWN TOOL
            problems.append(f"{n.eid}: unknown tool '{n.tool}' (not in the registry)")
        for d in n.deps:
            if d not in ids:                                        # UNSATISFIABLE DEP
                problems.append(f"{n.eid}: depends on '{d}', which is not in the plan")
        key = (n.tool, n.arg)
        if key in seen:                                             # REDUNDANT
            problems.append(f"{n.eid}: redundant, identical to {seen[key]} ({n.tool}({n.arg!r}))")
        else:
            seen[key] = n.eid
    if _has_cycle(plan):                                            # DEPENDENCY CYCLE
        problems.append("dependency cycle detected (a step transitively depends on itself)")
    return problems


print("critic and _has_cycle ready.")

## Step 4 -- `generate()`: the real LLM path (reference shape only)

Same device as Parts 1-3. In production the planner, critic, and replanner are all an LLM: you hand it the goal, the current plan, and the failure, and it returns a revised plan in the same `TaskNode` shape. `generate()` is the single swappable call that lights up the real path; the offline demo never calls it. The deterministic rule planner and critic are the source of truth for everything you see in the output. `_fmt` is a tiny display helper that prints floats as dollar amounts. SDK names and model IDs move fast, so only `generate()` would need edits to go live.

In [ ]:
def generate(prompt):
    """REAL path: ask a hosted LLM to plan, critique, or replan. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


def _fmt(value):
    if isinstance(value, float):
        return f"${value:.2f}"
    return f"{value}"


if os.environ.get("OPENAI_API_KEY"):
    print("[planner] OPENAI_API_KEY set; the real LLM would plan/critique/replan via "
          "generate(). Falling through to the deterministic rules so output is reproducible.")
else:
    print("[planner] no OPENAI_API_KEY; using deterministic rule planner/critic (offline default)")

## Step 5 -- The BLIND executor (Part 3, unchanged): the world disagrees

This is the Part 3 DAG executor with no critic and no replanner, here to make the failure concrete. It runs the plan in topological order, binding `#En` references as it goes. When `get_price` and `get_warranty` raise `DiscontinuedError`, it does the Part 2 thing: it catches the permanent error and feeds it back as an observation. That is honest, and it is not enough. The plan still names `SKU-ACME-EB` in three places, so `E2` and `E3` never produce a value, `E4`'s tax expression still contains the unresolved `#E2` and the calculator returns its error string, and the final quote is **INCOMPLETE**. The executor *knew* the SKU was dead and still could not revise the committed plan: feeding the error back told it the step failed, but gave it no way to act on a plan already written.

In [ ]:
def run_blind(plan):
    memo = {}
    for n in topo_order(plan):
        arg = _bind_arg(n.arg, memo)
        try:
            result = REGISTRY[n.tool](arg)
            memo[n.eid] = result
            print(f"    {n.eid}: {n.tool}({arg!r}) -> {_fmt(result)}")
        except DiscontinuedError as exc:
            print(f"    {n.eid}: {n.tool}({arg!r}) -> PermanentError: {exc}")
            print(f"         fed back as an observation (Part 2), but the plan still names a dead SKU")
    price = memo.get("E2")
    warranty = memo.get("E3")
    total = memo.get("E4")
    print("    QUOTE: INCOMPLETE -- "
          f"price={'unavailable' if price is None else _fmt(price)}, "
          f"warranty={'unavailable' if warranty is None else warranty}, "
          f"total={'uncomputable' if not isinstance(total, (int, float)) else _fmt(total)}.")
    print("    -> The executor knew the SKU was dead and still could not revise the committed plan.")
    return memo


print("run_blind ready.")

## Step 6 -- The CRITIC + REPLANNER executor: clear the plan, then rewrite only the tail

This is the second net-new mechanism, and it wraps the same DAG executor. First it runs the critic; if the plan has any structural problem it is **rejected before execution** and nothing fires. A cleared plan then executes by topological order, but with two additions. **Memoization**: completed nodes live in `memo`, and a node already present there is skipped, never re-run. **Error-triggered partial replanning**: when a node raises `DiscontinuedError`, the executor does not abandon the run and does not start over. It takes the not-yet-completed tail (`order[i:]`), rewrites only those nodes whose argument mentions the dead SKU to the replacement, leaves the memoized nodes untouched, and retries the *same* node with its revised argument (it does not advance `i`).

Two details earn their keep. `E4`'s argument is `#E2 * 1.18`, which names no SKU, so it is **not** in the rewritten tail; it simply binds the new price when its turn comes. And every replan is counted against `max_replans` (2 here), so a plan that keeps breaking trips the budget and stops rather than looping forever. The full circuit breaker is Part 8; this is the honest cap.

In [ ]:
def run_with_replanning(plan, max_replans=2):
    problems = critic(plan, REGISTRY)
    if problems:
        print("    critic REJECTED the plan before execution:")
        for p in problems:
            print(f"      - {p}")
        return None
    print("    critic: no problems found; plan cleared for execution.")

    order = topo_order(plan)
    memo = {}
    replans = 0
    i = 0
    while i < len(order):
        n = order[i]
        if n.eid in memo:                         # already completed: memoized, skip
            i += 1
            continue
        arg = _bind_arg(n.arg, memo)
        try:
            result = REGISTRY[n.tool](arg)
            memo[n.eid] = result
            print(f"    {n.eid}: {n.tool}({arg!r}) -> {_fmt(result)}")
            i += 1
        except DiscontinuedError as exc:
            if replans >= max_replans:
                print(f"    {n.eid}: DiscontinuedError again; replan budget ({max_replans}) "
                      "exhausted -> stop (the circuit breaker is Part 8).")
                return None
            replans += 1
            tail = order[i:]                       # the not-yet-completed subgraph
            revised = [m.eid for m in tail if exc.sku in m.arg]
            for m in tail:
                if exc.sku in m.arg:
                    m.arg = m.arg.replace(exc.sku, exc.replacement)
            kept = [e for e in memo]               # completed nodes stay put
            print(f"    {n.eid}: {n.tool}({arg!r}) -> PermanentError: {exc}")
            print(f"         REPLAN #{replans}: rewrite remaining {revised} to {exc.replacement}; "
                  f"memoized {kept} stay (not re-run).")
            # do not advance i: retry this node with its revised argument
    price, warranty, total = memo["E2"], memo["E3"], memo["E4"]
    final_sku = order[1].arg                       # E2's (possibly revised) SKU
    print(f"    QUOTE: {final_sku} (replaces discontinued SKU-ACME-EB): price {_fmt(price)}, "
          f"{warranty}, total with tax {_fmt(total)}.")
    print(f"    -> Completed with {replans} replan(s); the dead tail was rewritten, the lookup reused.")
    return memo


print("run_with_replanning ready.")

## Demo -- the critic, then the blind executor, then the replanner

Everything below runs fully offline. The demo has three sections. First, **the critic**: run it on a deliberately broken first draft (six nodes carrying all four kinds of mistake) so it reports exactly the four problems and rejects the plan, then run it on the real quote plan and watch it clear. The bad draft packs all four mistakes into six nodes: `B2` duplicates `B1` (REDUNDANT), `B3` calls a tool not in the registry (UNKNOWN TOOL), `B4` depends on `B9`, which does not exist (UNSATISFIABLE DEP), and `B5` and `B6` depend on each other (DEPENDENCY CYCLE). Second, **the blind executor** on the real plan: it hits the discontinued SKU, feeds the error back step by step exactly as Part 2 taught, and still cannot produce a valid quote, because it has no machinery to revise a committed plan. Third, **the critic + replanner** on the same plan: it clears the plan, hits the same failure, rewrites only the dead tail to `SKU-GLX-EB`, reuses the memoized lookup, and finishes with a correct quote and one replan.

In [ ]:
bar = "=" * 72
print(bar)
print("SURVIVING A BROKEN PLAN  -  the prospective critic and the replanner")
print(bar)
if os.environ.get("OPENAI_API_KEY"):
    print("[planner] OPENAI_API_KEY set; the real LLM would plan/critique/replan via "
          "generate(). Falling through to the deterministic rules so output is reproducible.")
else:
    print("[planner] no OPENAI_API_KEY; using deterministic rule planner/critic (offline default)")
print(f"\nGOAL: {GOAL}")

# --- The prospective critic, on a deliberately broken plan. -----------------
print("\n" + "-" * 72)
print("THE CRITIC: catch a bad plan BEFORE it wastes a single tool call.")
print("-" * 72)
bad_plan = [
    TaskNode("B1", "get_price", "SKU-GLX-EB", deps=[]),
    TaskNode("B2", "get_price", "SKU-GLX-EB", deps=[]),          # redundant with B1
    TaskNode("B3", "search_web", "earbuds review", deps=[]),     # unknown tool
    TaskNode("B4", "calculator", "#B9 * 2", deps=["B9"]),        # depends on missing B9
    TaskNode("B5", "get_warranty", "SKU-GLX-EB", deps=["B6"]),   # B5 <-> B6 cycle
    TaskNode("B6", "lookup_status", "SKU-GLX-EB", deps=["B5"]),
]
print("  Critiquing a planner's first draft (it has four kinds of mistake):")
for p in critic(bad_plan, REGISTRY):
    print(f"      - {p}")
print("  -> rejected; the planner is asked to try again before anything runs.")

print("\n  The same critic on the real quote plan:")
good_plan = rule_planner("SKU-ACME-EB")
lv = dag_levels(good_plan)
print("  plan:  " + " | ".join(f"{n.eid}={n.tool}({n.arg!r})" for n in good_plan))
print("  levels: " + ", ".join(f"{eid}@L{lv[eid]}" for eid in sorted(lv)))
gp = critic(good_plan, REGISTRY)
print(f"  critic: {'no problems found; cleared for execution.' if not gp else gp}")

# --- Blind executor: knows the SKU is dead, still cannot recover. -----------
print("\n" + bar)
print("BLIND EXECUTOR (Part 3 DAG, no critic, no replanner): the world disagrees.")
print(bar)
run_blind(rule_planner("SKU-ACME-EB"))

# --- Critic + replanner: revise only the dead tail, reuse the rest. ---------
print("\n" + bar)
print("CRITIC + REPLANNER: clear the plan, hit the failure, rewrite only the tail.")
print(bar)
run_with_replanning(rule_planner("SKU-ACME-EB"))

print("\n" + bar)
print("Done. An up-front plan is a bet on a world that can change under you:")
print("  - a prospective CRITIC rejects a structurally broken plan before it runs")
print("  - an error-triggered REPLANNER revises only the remaining subgraph on failure")
print("  - completed steps are MEMOIZED (never re-run); a replan BUDGET stops the loop")
print("Feeding an error back (Part 2) says the step failed; replanning is how a")
print("committed plan does something about it.")
print(bar)

## Wrap-up

Part 3 made the plan a first-class artifact, and that bought a new failure: an up-front plan is a bet on a world that can change under you, and the DAG executor charges ahead even when a SKU goes discontinued mid-run. This part added two mechanisms around the unchanged DAG machinery:

- a **prospective CRITIC** that rejects a structurally broken plan (unknown tool, unsatisfiable dependency, dependency cycle, redundant step) before a single tool fires, the plan-time analog of Part 1's argument validator;
- an **error-triggered REPLANNER** that, on a plan-invalidating failure, revises **only the remaining subgraph** to the replacement, keeps completed steps **MEMOIZED** so they are never re-run, and continues under a **replan BUDGET** that stops a plan that keeps breaking.

The contrast was the whole point: the blind executor knew the SKU was dead and still could not recover, while the critic + replanner cleared the plan, rewrote only the dead tail, reused the memoized lookup, and finished with a correct quote. Feeding an error back (Part 2) says the step failed; replanning is how a committed plan does something about it.

**Part 5 -- Reflection** is next: the critic here judged the plan's *structure* before it ran, but a plan can be structurally perfect and still produce a bad *answer*. Reflection turns the agent's judgment on its own output, letting it critique and revise a result that ran without error but is still wrong.